# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. Each `record_set`, `field`, and `column` is uniquely identified by its `@id` (usually a URI or string in the Croissant schema).

In [ ]:
# List all Record Sets and their available fields by `@id`
record_sets = list(dataset.record_sets)
print(f"Available record sets ({len(record_sets)}):")
for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields (by @id):")
    for f in rs.fields:
        print(f"    - {f.id} ({f.name})")
    print("")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above. For this example, we select the first available record set.

In [ ]:
# Extract data from all available record sets using their @id
dataframes = {}
for rs in dataset.record_sets:
    record_set_id = rs.id
    print(f"Loading records for record set '{record_set_id}'...")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {list(df.columns)}\n  Sample records:\n{df.head(2)}\n")
    else:
        print("  No records found. Skipping.\n")

# Select the first non-empty record set for further exploration
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    main_df = dataframes[main_record_set_id]
    print(f"Selected record set for EDA: {main_record_set_id}")
else:
    main_record_set_id = None
    print("No record sets with records were found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, or grouping. We'll select a numeric field by its `@id`, and demonstrate standard EDA procedures. Update `<numeric_field_id>` as relevant for your data.

In [ ]:
import numpy as np

if main_record_set_id is not None:
    # Let's attempt to pick a typical numeric field from proteomics tables,
    # e.g. 'molecular_weight', 'abundance', 'coverage', etc.
    sample_columns = main_df.columns.tolist()
    print(f"Columns in '{main_record_set_id}':\n  {sample_columns}")

    # Try to pick a numeric field (`@id`) that is present in the DataFrame
    # You should update this if you know a specific `@id` from the earlier overview
    numeric_candidates = [
        col for col in sample_columns if any(keyword in col.lower() for keyword in ['abundance', 'weight', 'coverage', 'count', 'amount', 'score', 'total'])
    ]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Use the first match
        print(f"Using numeric field: {numeric_field}")
    else:
        # Default to the first column if no match found
        numeric_field = sample_columns[0]

    # 1. Remove outliers (example: numeric_field > threshold)
    try:
        col_as_num = pd.to_numeric(main_df[numeric_field], errors='coerce')
        threshold = col_as_num.quantile(0.1)
        filtered_df = main_df[col_as_num > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (10th percentile of non-NaN values): {len(filtered_df)} records")

        # 2. Normalize the numeric field
        mean = col_as_num.mean()
        std = col_as_num.std()
        filtered_df[f"{numeric_field}_normalized"] = (pd.to_numeric(filtered_df[numeric_field], errors='coerce') - mean) / std
        print(f"First few normalized values for {numeric_field}:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # 3. If another grouping field exists, group by it
        group_candidates = [
            col for col in sample_columns if col != numeric_field and main_df[col].dtype in [object, 'category']
        ]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            group_field = None
    except Exception as e:
        print(f"Could not perform numeric analysis due to: {str(e)}")
else:
    filtered_df = None
    numeric_field = None
    group_field = None

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll create a histogram for the selected numeric field, and if grouping is possible, a bar plot of group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if filtered_df is not None and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(pd.to_numeric(filtered_df[numeric_field], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group_field is available, plot mean by group
    if group_field:
        group_means = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        plt.figure(figsize=(10, 4))
        group_means.plot(kind='bar')
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've outlined how to use `mlcroissant` to:
- Load and inspect dataset metadata and schema using a Croissant URL
- Enumerate record sets and their fields by `@id` for consistent reference
- Load record set data into pandas DataFrames
- Perform basic filtering, normalization, and exploratory grouping based on selected fields
- Visualize numeric data distributions, and examine relationships by grouping fields

This approach can be extended for deeper analysis, more advanced statistical methods, and integration with downstream ML workflows. Always use field and record set `@id`s for reproducibility when using Croissant datasets.